# 05 - Other TWPAs

The previous tutorials used a three-wave-mixing kinetic-inductance TWPA (KI-TWPA). This notebook builds two other
designs with the same tools:

1. **Resonant phase matching** in a four-wave-mixing (4WM) Josephson TWPA: LC resonators periodically inserted in the
   shunt branch of the line create a narrow notch in the transmission, and the dispersion just below it compensates
   the Kerr phase mismatch of the pump. We follow the design principles of C. Denney *et al.* (M. Malnou's group),
   *A Traveling-Wave Parametric Amplifier With Integrated Diplexers*, arXiv:2603.12327 (2026); the technique goes back
   to K. O'Brien *et al.*, PRL **113**, 157001 (2014) and C. Macklin *et al.*, Science **350**, 307 (2015). The cell is a
   user-defined model, which also shows how to make a custom cell work with the `bloch` model.
2. **A Floquet-mode TWPA built from segments**: the nonlinearity is switched on and off smoothly along the line, so
   that the Floquet modes of the pumped line connect adiabatically to the linear modes at both ends and less signal is
   lost into frequency-conversion sidebands. The idea is from K. Peng *et al.*, *Floquet-Mode Traveling-Wave Parametric
   Amplifiers*, PRX Quantum **3**, 020306 (2022) (first experiment: arXiv:2503.11812, 2025). We apply it to the KI-TWPA
   of the previous tutorials, cut into segments with different nonlinearity.

In [ ]:
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np

from twpasolver import ModeSet, TWPAnalysis
from twpasolver.cmes.bloch import PORT_IN
from twpasolver.cmes.solver_config import BlochSolverConfig
from twpasolver.logger import log
from twpasolver.matrices_arrays import ABCDArray
from twpasolver.models import TWPA
from twpasolver.twoport import TwoPortModel

log.setLevel("WARNING")

## 1. Resonant phase matching with notch resonators

### A Josephson cell as a user model

Each unit cell is a series Josephson junction, a linear inductance $L_J = \Phi_0 / (2\pi I_c)$ to first order, and a
shunt capacitor $C$ to ground. In the resonator cells the shunt capacitor reaches ground through a parallel LC tank
$(L_r, C_r)$. A new two-port model only needs a `name` field and a `single_abcd` method; the method
`bloch_nl_elements()` tells the `bloch` model which series inductors are nonlinear: here the junction, which carries
the current of the cell's input port. Because the class declares its own `name`, cell lists and JSON files can also
refer to it by name (see *Implementing New Models*).

In [ ]:
class JJCell(TwoPortModel):
    """Series junction (linear inductance L), shunt C; optional parallel LC tank in series with C."""

    name: Literal["JJCell"] = "JJCell"
    L: float
    C: float
    Lr: float = 0.0
    Cr: float = 0.0

    def single_abcd(self, freqs):
        w = 2 * np.pi * np.asarray(freqs, dtype=float)
        Z_series = 1j * w * self.L
        Z_shunt = 1 / (1j * w * self.C)
        if self.Cr > 0:
            Z_shunt = Z_shunt + 1 / (1 / (1j * w * self.Lr) + 1j * w * self.Cr)
        Y = 1 / Z_shunt
        abcd = np.empty((len(w), 2, 2), dtype=complex)
        abcd[:, 0, 0] = 1 + Z_series * Y
        abcd[:, 0, 1] = Z_series
        abcd[:, 1, 0] = Y
        abcd[:, 1, 1] = 1
        return ABCDArray(abcd)

    def bloch_nl_elements(self):
        return [(PORT_IN, self.L)]

### Device parameters

From the paper: a 50 $\Omega$ line of junctions with $I_c \simeq 5\,\mu$A, resonators at 8.7 GHz with a low mode
impedance $Z_r \approx 15\,\Omega$ (a narrow stopband), and a pump near 8.5 GHz with the signal band below 8 GHz. The
paper does not list the cell count or the resonator spacing, so we **assume** one resonator every 4 cells and 2000
cells. The junction nonlinearity $L_J(I) \simeq L_J\,[1 + I^2 / (2 I_c^2)]$ is twpasolver's $L(1 + I^2/I_*^2)$ with
$I_* = \sqrt{2} I_c$; without dc bias ($I_{dc}=0$) only 4WM is present. The integrated diplexers of the paper (a
three-port circuit) are not modelled. Near the notch the wavenumber changes quickly within a cell, and the library
may warn that its two estimates of the Bloch phase differ by a few mrad: harmless at that size.

In [ ]:
PHI0 = 2.067833848e-15
Ic, Z0 = 5e-6, 50.0
LJ = PHI0 / (2 * np.pi * Ic)
C = LJ / Z0**2  # 50 Ohm line
fr, Zr = 8.7e9, 15.0
Lr, Cr = Zr / (2 * np.pi * fr), 1 / (2 * np.pi * fr * Zr)
period, n_cells = 4, 2000  # assumed

print(
    f"L_J = {LJ * 1e12:.1f} pH, C = {C * 1e15:.1f} fF, L_r = {Lr * 1e9:.3f} nH, C_r = {Cr * 1e12:.3f} pF"
)


def jj_line(resonators: bool, Ip0: float) -> TWPA:
    if resonators:
        cells = [JJCell(L=LJ, C=C, N=period - 1), JJCell(L=LJ, C=C, Lr=Lr, Cr=Cr)]
    else:
        cells = [JJCell(L=LJ, C=C, N=period)]
    return TWPA(
        cells=cells,
        N=n_cells // period,
        Istar=np.sqrt(2) * Ic,
        Idc=0.0,
        Ip0=Ip0,
        Z0_ref=Z0,
    )


Ip0 = 0.6 * Ic
rpm = TWPAnalysis(twpa=jj_line(True, Ip0), freqs=(0.5, 20, 2e-3))
plain = TWPAnalysis(twpa=jj_line(False, Ip0), freqs=(0.5, 20, 2e-3))
print("stopbands with resonators (GHz):", rpm.linear.stopbands)

### Linear response: the notch

Just below the tank resonance, the inductive tank cancels the reactance of the shunt capacitor and the shunt branch
becomes a short: a narrow stopband (the notch) opens at about 8.61 GHz. Below it, the wavenumber bends upward. $k^*$
removes the linear part of $k$, so the bend is the extra phase the resonators give the pump.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), sharex=True)
for an, label in ((plain, "no resonators"), (rpm, "resonators")):
    lin = an.linear
    ax1.plot(lin.freqs, lin.S21_db, label=label)
    ax2.plot(lin.freqs, lin.k_star, label=label)
for ax in (ax1, ax2):
    ax.axvspan(*rpm.linear.stopbands[0], color="gray", alpha=0.3)
    ax.legend()
ax1.set_ylabel("$|S_{21}|$ (dB)")
ax1.set_ylim(-40, 2)
ax2.set_ylabel("$k^*$ (rad/cell)")
ax2.set_xlabel("Frequency (GHz)")
ax2.set_xlim(2, 12)
plt.show()

### Phase matching

For degenerate 4WM ($2 f_p = f_s + f_i$) the mismatch includes the Kerr terms of the pump, which grow with the pump power. A pump placed just below the notch
recovers the missing wavenumber: over the 4-8 GHz band the resonators halve the best mean relative mismatch, and the
optimal pump moves next to the notch (without them it sits at the edge of the scanned range). `phase_matching` maps
the relative mismatch over pump and signal frequencies.

In [ ]:
pumps = np.arange(7.6, 8.6, 0.01)
pm_rpm = rpm.phase_matching((4, 8, 0.1), pumps, mixing="4WM", modes="basic_4wm")
pm_plain = plain.phase_matching((4, 8, 0.1), pumps, mixing="4WM", modes="basic_4wm")
for name, pm in (("resonators", pm_rpm), ("no resonators", pm_plain)):
    best = np.nanmin(np.nanmean(np.abs(pm.delta), axis=0))
    print(
        f"{name}: best pump {pm.optimal_pump_freq:.2f} GHz, mean |delta| there {best:.2e}"
    )
pm_rpm.plot()
plt.show()

### Gain

At the same pump amplitude ($0.6\,I_c$) the resonators raise the gain by about 10 dB. The `simple` and `bloch` models
agree: this line is matched to 50 $\Omega$ and the notch is far from the band. The gain collapses where the idler
$f_i = 2 f_p - f_s$ falls inside the notch and cannot propagate.

In [ ]:
fp = 8.12
signals = np.linspace(3.5, 8.2, 48)
fig, ax = plt.subplots(figsize=(6, 3.5))
for an, label in ((rpm, "resonators"), (plain, "no resonators")):
    for model, style in (("simple", "-"), ("bloch", "--")):
        r = an.gain(signals, pump=fp, Is0=1e-9, model=model, modes="basic_4wm")
        ax.plot(r.signal_freqs, r.gain_db, style, label=f"{label}, {model}")
lo, hi = rpm.linear.stopbands[0]
ax.axvspan(
    2 * fp - hi, 2 * fp - lo, color="gray", alpha=0.3, label="idler in the notch"
)
ax.set_xlabel("Signal frequency (GHz)")
ax.set_ylabel("Gain (dB)")
ax.legend(fontsize=8)
plt.show()

### Check the mode set

The basic mode set keeps only pump, signal and idler. In a Josephson line without dispersion at high frequency, the
pump third harmonic and the conversions $2f_p \pm$ signal or idler are nearly phase matched too, so adding them
changes the predicted gain a lot. The real device is tamed by the junction plasma frequency (the junction
capacitance, not modelled here). Always check that the result is stable when modes are added.

In [ ]:
extended = ModeSet(
    ["p", "s", "i", "p3", "p2s", "p2i"],
    [["i", "p+p-s"], ["p3", "p+p+p"], ["p2s", "p+p+s"], ["p2i", "p+p+i"]],
)
print(extended)
sig = np.linspace(4, 7.4, 8)
for modes in ("basic_4wm", extended):
    r = rpm.gain(sig, pump=fp, Is0=1e-9, model="bloch", modes=modes)
    name = "basic" if isinstance(modes, str) else "with sidebands"
    print(f"{name}: mean gain {np.mean(r.gain_db):.1f} dB")

## 2. A Floquet-mode TWPA from segments

In a uniformly pumped TWPA the signal does not only turn into idler photons: the pump also converts it to sidebands
(here $f_p \pm f_s$, $2 f_p \pm f_s$ and the corresponding idler terms). Photons left in those modes at the output
are lost information, which lowers the quantum efficiency. Peng *et al.* showed that switching the nonlinearity on
and off adiabatically along the line maps the input and output linear modes onto single Floquet modes of the pumped
line, so that the sideband content at the output nearly vanishes. The taper does not phase match anything; it cleans
the output.

We take the KI-TWPA of the previous tutorials, cut it into 40 segments, and scale its nonlinearity
($\propto 1/I_*^2$) with a Gaussian profile that keeps 10 % at the ends. The linear circuit is the same in every
segment, so the taper adds no reflections. Each segment holds a whole number of supercells, as segmented devices
require.

In [ ]:
base = TWPA.from_file("imskit_v3_holotype.json")
n_seg, floor, sigma = 40, 0.1, 0.2
u = (np.arange(n_seg) + 0.5) / n_seg
profile = floor + (1 - floor) * np.exp(-((u - 0.5) ** 2) / (2 * sigma**2))


def floquet_line(Ip0: float) -> TWPA:
    segments = [
        base.model_copy(
            deep=True,
            update={"N": base.N // n_seg, "Istar": base.Istar / np.sqrt(p), "Ip0": Ip0},
        )
        for p in profile
    ]
    return TWPA(cells=segments, Z0_ref=base.Z0_ref, Ip0=Ip0)


plt.figure(figsize=(5, 2.5))
plt.step(u, profile, where="mid")
plt.xlabel("Position along the line")
plt.ylabel("Relative nonlinearity")
plt.show()

The weaker average nonlinearity is compensated by a stronger pump (0.181 mA instead of 0.1 mA), which gives the
same mean gain. We keep the pump harmonics and two orders of frequency conversion, solve with the `bloch` model and
count, at the output, the photon flux $\propto |I|^2/f$ in the conversion modes relative to signal plus idler.

In [ ]:
EXT = ModeSet.extended_3wm(
    n_pump_harmonics=3, n_frequency_conversion=2, n_signal_harmonics=0
)
LOST = ("ps", "pi", "p2s", "p2i")
signals = np.linspace(4, 11, 15)
fp, cfg = 15.8, BlochSolverConfig(passes=60)
results = {}
for name, twpa in (
    ("uniform", base.model_copy(update={"Ip0": 1e-4})),
    ("Floquet", floquet_line(1.809e-4)),
):
    an = TWPAnalysis(twpa=twpa, freqs=(0.2, 80, 2e-3))
    r = an.gain(signals, pump=fp, Is0=1e-9, model="bloch", modes=EXT, bloch_config=cfg)
    f = an.problem(EXT, {"p": fp, "s": signals}).freqs
    flux = np.abs(r.b_out) ** 2 / np.abs(f)
    lost = sum(flux[:, r.mode(m)] for m in LOST)
    useful = flux[:, r.mode("s")] + flux[:, r.mode("i")]
    results[name] = (r, lost / useful)
    print(
        f"{name}: mean gain {np.mean(r.gain_db):.1f} dB, lost / (signal + idler) photons "
        f"{10 * np.log10(np.mean(lost / useful)):.1f} dB, converged {r.converged.sum()}/{len(signals)}"
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
for name, (r, ratio) in results.items():
    ax1.plot(r.signal_freqs, r.gain_db, label=name)
    ax2.plot(r.signal_freqs, 10 * np.log10(ratio), label=name)
ax1.set_ylabel("Gain (dB)")
ax2.set_ylabel("Lost / useful photons (dB)")
for ax in (ax1, ax2):
    ax.set_xlabel("Signal frequency (GHz)")
    ax.legend()
plt.tight_layout()
plt.show()

At equal mean gain the tapered line leaves about 20 times (13 dB) fewer photons in the conversion modes. (One
signal point stops just above the convergence threshold in both runs; more passes give a cleaner curve.) The mode
currents along the line show why: in the uniform device the conversion modes keep growing up to the output, while
in the Floquet device they are populated in the middle and return to the signal and idler as the nonlinearity
switches off.

In [ ]:
for name, (r, _) in results.items():
    ax = r.plot_currents(title=f"{name}: mode currents, averaged over the signals")
plt.show()